# Sampling Activations from InsanallyAlbanna SNN

Post-processes simulation output from the **InsanallyAlbanna-NatComm-2024** spiking neural network
(Insanally & Albanna, Nature Communications 2024) into the `tensor4d` format used by this pipeline.

This model is a **leaky integrate-and-fire (LIF) recurrent network** trained on a frequency-discrimination task
using FORCE + STDP learning rules.  Stimuli are discrete **frequency values** rather than visual stimuli;
the manifold comparison is non-visual but valid — neurons, stimuli, and time are all present.

## Setup (run once before this notebook)

```bash
# 1. Clone the repo (e.g. into ~/repos/ or a sibling of this repo)
git clone https://github.com/albannalab/InsanallyAlbanna-NatComm-2024
cd InsanallyAlbanna-NatComm-2024

# 2. Install Julia (https://julialang.org/downloads/) then:
julia --project=. -e 'using Pkg; Pkg.instantiate()'

# 3. Run the example simulation — produces HDF5 output (~30 min on CPU)
julia --project=. example_simulation.jl
```

The simulation writes HDF5 files to the repo's output directory.
Set `SNN_REPO_PATH` and `SNN_OUTPUT_FILE` below to point to those files.

## Output
`data/sampled/tensor4d_insanally_i<N>_n<N>_seed17.npy`  
Shape: `(N_neurons, N_freq_stimuli, 1, N_time_bins)` — NDIRS=1 (no directional component).

In [ ]:
import numpy as np
import h5py
import sys, os

_d = os.path.abspath(os.getcwd())
sys.path.insert(0, _d if os.path.isdir(os.path.join(_d, 'src')) else os.path.dirname(_d))

# --- Paths to InsanallyAlbanna repo and its output ---
SNN_REPO_PATH   = os.path.expanduser('~/repos/InsanallyAlbanna-NatComm-2024')  # <-- update
SNN_OUTPUT_FILE = os.path.join(SNN_REPO_PATH, 'output', 'simulation_output.h5')  # <-- update to actual filename

# Add the repo's Python helpers to the path
sys.path.insert(0, SNN_REPO_PATH)

assert os.path.isfile(SNN_OUTPUT_FILE), (
    f'HDF5 output not found at {SNN_OUTPUT_FILE}. '
    'Run the Julia simulation first (see notebook header).'
)

print('SNN output file found:', SNN_OUTPUT_FILE)

In [ ]:
########## EXPLORE HDF5 STRUCTURE ##########
# Run this cell first to understand what keys the HDF5 file contains.

def print_hdf5_tree(f, indent=0):
    for key in f.keys():
        item = f[key]
        if hasattr(item, 'shape'):
            print(' ' * indent + f'{key}: shape={item.shape}, dtype={item.dtype}')
        else:
            print(' ' * indent + f'{key}/')
            print_hdf5_tree(item, indent + 2)

with h5py.File(SNN_OUTPUT_FILE, 'r') as f:
    print_hdf5_tree(f)

In [ ]:
########## PARAMS ##########
# Adjust these after inspecting the HDF5 structure above.

# Keys inside the HDF5 file — update to match actual structure
SPIKE_DATA_KEY    = 'activity'           # key containing spike data
STIMULUS_KEY      = 'stimulus_tr'        # key containing per-trial stimulus frequency value
TRIAL_DURATION_MS = 200                  # ms per trial (inspect from simulation config)

# Temporal binning
BIN_SIZE_MS       = 5                    # ms per time bin
N_TIME_BINS       = TRIAL_DURATION_MS // BIN_SIZE_MS  # e.g., 40 bins

# Frequency stimulus discretisation
# The simulation presents tones at various frequencies; group them into bins.
# Inspect the stimulus_tr array to set these.
N_FREQ_STIMULI    = None  # set after inspecting stimulus values

seed              = 17

# For the tensor4d format, NDIRS=1 (no directional component)
NDIRS             = 1

print(f'Planned tensor shape: (N_neurons, {N_FREQ_STIMULI}, {NDIRS}, {N_TIME_BINS})')

In [ ]:
########## LOAD SPIKE DATA + INSPECT FREQUENCIES ##########

# Try using the repo's own I/O helper if available
try:
    from RNN_helpers.io import load_activity
    spike_data = load_activity(SNN_OUTPUT_FILE)
    print('Loaded via RNN_helpers.io.load_activity')
except ImportError:
    # Fallback: load directly from HDF5
    with h5py.File(SNN_OUTPUT_FILE, 'r') as f:
        spike_data  = f[SPIKE_DATA_KEY][:]   # (n_spikes, 3): (trial_id, neuron_id, spike_time_ms)
        stim_values = f[STIMULUS_KEY][:]     # (n_trials,): frequency value per trial
    print('Loaded directly from HDF5')

print(f'spike_data shape : {np.array(spike_data).shape}')
print(f'stim_values shape: {np.array(stim_values).shape}')

# Inspect unique frequency values
unique_freqs = np.unique(stim_values)
print(f'Unique stimulus frequencies ({len(unique_freqs)} total):', unique_freqs)

In [ ]:
########## DEFINE STIMULUS CATEGORIES ##########
# Map continuous frequency values to discrete stimulus indices.
# Option A: treat each unique frequency as its own stimulus.
# Option B: bin frequencies into N_FREQ_STIMULI coarser categories.

# Option A (recommended if N unique freqs is manageable, e.g. < 30)
FREQ_CATEGORIES = sorted(unique_freqs.tolist())   # list of N_FREQ_STIMULI distinct values
N_FREQ_STIMULI  = len(FREQ_CATEGORIES)
freq_to_idx     = {f: i for i, f in enumerate(FREQ_CATEGORIES)}
trial_stim_idx  = np.array([freq_to_idx[v] for v in stim_values])  # (n_trials,)

print(f'N_FREQ_STIMULI = {N_FREQ_STIMULI}')
print(f'Trials per stimulus: {np.bincount(trial_stim_idx)}')

In [ ]:
########## PARSE SPIKE TIMES → TRIAL-ALIGNED RASTERS ##########
# Try the repo's parse_activity helper; fall back to manual.

try:
    from RNN_helpers.analysis import parse_activity
    rasters = parse_activity(spike_data)   # inspect returned structure
    print('Parsed via RNN_helpers.analysis.parse_activity')
    print('rasters type:', type(rasters))
    # Adapt the raster format to the binning step below based on what parse_activity returns.
except (ImportError, Exception) as e:
    print(f'parse_activity not available ({e}), building rasters manually.')
    rasters = None

# Convert spike tuples to (trial_id, neuron_id, spike_time_ms) array
spikes = np.array(spike_data)             # (n_spikes, 3)
if spikes.ndim == 1 and len(spikes) > 0 and hasattr(spikes[0], '__len__'):
    spikes = np.stack(spikes)             # handle list-of-tuples

trial_ids   = spikes[:, 0].astype(int)
neuron_ids  = spikes[:, 1].astype(int)
spike_times = spikes[:, 2].astype(float)

n_trials  = len(stim_values)
N_neurons = neuron_ids.max() + 1
print(f'n_trials={n_trials}, N_neurons={N_neurons}')

In [ ]:
########## BIN SPIKES → FIRING RATE TRACES ##########
# For each (neuron, stimulus_category): average PSTH across trials.

bin_edges = np.arange(0, TRIAL_DURATION_MS + BIN_SIZE_MS, BIN_SIZE_MS)  # (N_TIME_BINS+1,)
assert len(bin_edges) - 1 == N_TIME_BINS

# Accumulators: spike counts and trial counts per (neuron, stim_category, time_bin)
spike_counts = np.zeros((N_neurons, N_FREQ_STIMULI, N_TIME_BINS), dtype='float32')
trial_counts = np.zeros((N_neurons, N_FREQ_STIMULI), dtype='int')

# Accumulate spike counts
for tr_idx in range(n_trials):
    stim_i   = trial_stim_idx[tr_idx]
    mask     = trial_ids == tr_idx
    nids     = neuron_ids[mask]
    times    = spike_times[mask]

    for nid, t in zip(nids, times):
        bin_idx = int(t // BIN_SIZE_MS)
        if 0 <= bin_idx < N_TIME_BINS:
            spike_counts[nid, stim_i, bin_idx] += 1

    trial_counts[:, stim_i] += 1

# Average over trials → firing rate in spikes/bin; convert to Hz
n_trials_per_stim = trial_counts[0]  # same for all neurons
firing_rates = spike_counts / np.maximum(n_trials_per_stim[None, :, None], 1)  # (N, S, T)
firing_rates_hz = firing_rates * (1000.0 / BIN_SIZE_MS)

print(f'firing_rates_hz shape: {firing_rates_hz.shape}')  # (N_neurons, N_FREQ_STIMULI, N_TIME_BINS)
print(f'Mean firing rate: {firing_rates_hz.mean():.2f} Hz')

In [ ]:
########## BUILD TENSOR4D ##########
# Shape: (N_neurons, N_FREQ_STIMULI, 1, N_TIME_BINS)
# NDIRS=1 — no directional component.

tensorX = firing_rates_hz[:, :, np.newaxis, :]  # (N, S, 1, T)
print(f'tensor4d shape: {tensorX.shape}')

# Filter out silent neurons
np.random.seed(seed)
max_rate_per_neuron = tensorX.max(axis=(1, 2, 3))  # (N,)
active_mask = max_rate_per_neuron > 0.0
tensorX = tensorX[active_mask]
active_ids = np.where(active_mask)[0]
N_neurons_kept = tensorX.shape[0]
print(f'Active neurons: {N_neurons_kept} / {len(active_mask)}')

In [ ]:
########## SAVE ##########

os.makedirs('../data/sampled', exist_ok=True)

N_INSTANCES_EQUIV = n_trials_per_stim.min()  # trials per stimulus ≈ instance count
SUFFIX = f'insanally_i{int(N_INSTANCES_EQUIV)}_n{N_neurons_kept}_seed{seed}'

out_tensor  = f'../data/sampled/tensor4d_{SUFFIX}.npy'
out_neurons = f'../data/sampled/neurons_used_{SUFFIX}.npy'

if os.path.exists(out_tensor):
    print(f'[SKIP] {out_tensor} already exists — delete to regenerate')
else:
    np.save(out_tensor, tensorX)
    np.save(out_neurons, active_ids)
    print(f'Saved {out_tensor}  shape={tensorX.shape}')
    print(f'Saved {out_neurons}')

print(f'\n--- Summary ---')
print(f'PREFIX for downstream notebooks: {SUFFIX}')
print(f'Shape: {tensorX.shape}  (N={N_neurons_kept}, S={N_FREQ_STIMULI}, D=1, T={N_TIME_BINS})')
print(f'Frequency stimuli ({N_FREQ_STIMULI}): {FREQ_CATEGORIES}')
print(f'Time bin size: {BIN_SIZE_MS} ms → T={N_TIME_BINS} bins over {TRIAL_DURATION_MS} ms')
print()
print('To add to notebook 06 (subpop sweep), add to _LAYER_PARAMS:')
print(f'  "{SUFFIX}": dict(low_sf=False),')

## Notes on downstream usage

- **NDIRS=1** — direction-selective metrics (OSI, preferred direction) will return zero/NaN. Expected.
- **CP decomposition** — with NDIRS=1 the permutation-alignment step in `getPermutedTensor` is skipped (returns data unchanged). The MATLAB step is still needed for `getNeuralMatrix` if you want encoding manifolds (nb03); alternatively skip nb03 and start from nb04/nb05/nb06 which do not require decomposition.
- **`process_tensor_data`** — the `optSF=True` path only applies for NSTIMS=11; with other NSTIMS values it falls through gracefully. Use `optSF=False`.
- **Stimulus semantics** — the 'stimuli' dimension is frequency categories, not visual patterns. Manifold geometry reflects frequency-tuning organisation rather than visual feature selectivity — a valid but different comparison.